In [ ]:
#| default_exp features.endmotif_1mer

In [ ]:
#| export
from __future__ import annotations
import pandas as pd
import structlog
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()
        

In [ ]:
#| export
def _parse_array(s):
    if not isinstance(s, str) or not s.startswith('['): 
        return []
    clean = s.replace('[', '').replace(']', '').replace(chr(10), '').replace(chr(13), '')
    try:
        return [float(x) for x in clean.split()]
    except:
        return []

class EndMotif1merEvaluator(FeatureEvaluator):
    """Extracts 1-mer fragment end frequencies."""
    
    name = "EndMotif1mer"
    source_file = ".EndMotif1mer.parquet"
    tier = 2
    category = "epigenetics_and_geometry"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted
            cols = set(df.columns)

            self.tier = 3
            self.category = "motifs"
            if "base" in cols and "fraction" in cols:
                for row in df.to_dict('records'):
                    b = str(row["base"])
                    if pd.notna(row["fraction"]):
                        extracted[f"base_1mer_{b}"] = float(row["fraction"])
    
            return extracted
        except Exception as e:
            log.warning("endmotif_1mer_extraction_failed", error=str(e))
            return {}


In [ ]:
#| test
evaluator = EndMotif1merEvaluator()
df = pd.DataFrame([{"base": "A", "fraction": 0.25}])
res = evaluator.extract(df)
assert res["base_1mer_A"] == 0.25
